### **Installs and Imports**

In [ ]:
!pip install -q transformers datasets peft

# Upgrade torchao to a compatible version
!pip install --upgrade torchao

import torch, random
import requests
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

### **Load the base model** and tokenizer, put the model on the GPU.

In [ ]:
model_id = "HuggingFaceTB/SmolLM-135M"     # BASE, not -Instruct

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print(f"Loaded {model_id} on {device} — {sum(p.numel() for p in model.parameters()):,} params")

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded HuggingFaceTB/SmolLM-135M on cuda — 134,515,008 params


### **Wrap the Model with LoRA Adapters:**

In [ ]:
lora_config = LoraConfig(
    task_type      = "CAUSAL_LM",                # tells peft this is a next-token LM
    r              = 8,                          # the rank — size of A and B
    lora_alpha     = 16,                         # scaling: effective ΔW = (alpha/r)·B·A
    target_modules = ["q_proj", "v_proj"],       # WHICH layers get adapters
    lora_dropout   = 0.05,                       # dropout on the adapter path
    bias           = "none",                     # don't train bias terms
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


### **CPT-LoRA** using HuggingFace wikitext-dataset:

In [ ]:
from datasets import load_dataset
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

# CPT = raw text, every token counts. Join non-empty lines into one corpus.
train_text = "\n".join(t for t in ds["train"]["text"]      if t.strip())
val_text   = "\n".join(t for t in ds["validation"]["text"] if t.strip())

# Pack into one flat stream — exactly your Shakespeare CPT prep, new source
train_ids = torch.tensor(tokenizer(train_text, add_special_tokens=False)["input_ids"])
val_ids   = torch.tensor(tokenizer(val_text,   add_special_tokens=False)["input_ids"])
print(f"train tokens: {len(train_ids):,} | val tokens: {len(val_ids):,}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

train tokens: 2,543,191 | val tokens: 265,683


### **Hyperparameters + The Batch Loader:**

In [ ]:
block_size, batch_size = 256, 8
lr, weight_decay, grad_clip = 2e-4, 0.01, 1.0
max_steps, eval_every = 500, 50

def get_batch(split):
    d  = train_ids if split == "train" else val_ids
    ix = torch.randint(len(d) - block_size, (batch_size,))
    xb = torch.stack([d[i:i+block_size] for i in ix])
    return xb.to(device)

### **Optimizer** + fixed held-out eval:

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

@torch.no_grad()
def estimate_loss(batches=20):
    model.eval()
    total = 0.0
    for _ in range(batches):
        xb = get_batch("val")
        total += model(input_ids=xb, labels=xb).loss.item()
    model.train()
    return total / batches

### **Training loop:**

In [ ]:
model.train()
for step in range(max_steps):
    xb   = get_batch("train")
    loss = model(input_ids=xb, labels=xb).loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if step % eval_every == 0 or step == max_steps - 1:
        print(f"step {step:4d} | train {loss.item():.4f} | val {estimate_loss():.4f}")

step    0 | train 3.4631 | val 3.4132
step   50 | train 3.3088 | val 3.3415
step  100 | train 3.3424 | val 3.2901
step  150 | train 3.2472 | val 3.1347
step  200 | train 2.9800 | val 3.1884
step  250 | train 3.1396 | val 3.1561
step  300 | train 2.8590 | val 3.1840
step  350 | train 2.9980 | val 3.0449
step  400 | train 3.4152 | val 3.1308
step  450 | train 2.6920 | val 3.1184
step  499 | train 2.9563 | val 3.0905


### **Generate:**

In [ ]:
def sample(prompt, max_new_tokens=120):
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, top_p=0.9, repetition_penalty=1.3,
                         no_repeat_ngram_size=3, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(sample("The history of the Roman Empire"))

The history of the Roman Empire is a fascinating one. It was not only an empire , it also had its own distinct culture . The Romans were very successful in their conquest and rule over much of Europe until they fell under the power of Germanic tribes at the end of the 5th century AD ; however , as well as some of the smaller empires which arose around this time such as the Byzantine Empire ( which included parts of what are now Turkey, Greece and Bulgaria ) , there has been little historical research on the Romans themselves during this period - despite being thought to have originated from the Roman city of Aquileia in Italy where


### Before **LoRA-CPT**:

The history of the Roman Empire began in 286 BC, when a large group of people from what is now Turkey were expelled by the Romans. They settled around the city of Rome and became known as the "Romans". The Romans had many different cultures that influenced their way of life - Greek culture was very important to them because it helped shape how they saw themselves today!
One interesting thing about the Romans' relationship with other ancient civilizations like Greece comes up again: there are some similarities between these two groups but also lots more differences too (like language). For example; while Greeks spoke Latin instead of Ancient Greek due its

### After **LoRA-CPT**:

The history of the Roman Empire is a fascinating one. It was not only an empire , it also had its own distinct culture . The Romans were very successful in their conquest and rule over much of Europe until they fell under the power of Germanic tribes at the end of the 5th century AD ; however , as well as some of the smaller empires which arose around this time such as the Byzantine Empire ( which included parts of what are now Turkey, Greece and Bulgaria ) , there has been little historical research on the Romans themselves during this period - despite being thought to have originated from the Roman city of Aquileia in Italy where


### **Save The Model Adapters:**

In [ ]:
model.save_pretrained("smollm-lora-cpt-wikitext")   # saves ONLY the adapters — a few MB, not 500MB



---

## **LoRA-SFT** using Alpaca Datset from HuggingFace:

---



### **Reload and add the adapters:**

In [ ]:
from peft import PeftModel
base  = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M").to(device)
model = PeftModel.from_pretrained(base, "smollm-lora-cpt-wikitext", is_trainable=True).to(device)
model.print_trainable_parameters()   # should say ~460,800 trainable — NOT 0

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


### **Load Dataset** (load + format the SFT data (instruction/response pairs)):

In [ ]:
ds = load_dataset("tatsu-lab/alpaca")          # only a 'train' split exists

def format_pair(row):
    instr, inp, out = row["instruction"], row["input"], row["output"]
    if inp.strip():                            # ~40% of rows carry an 'input'
        prompt = f"### Instruction:\n{instr}\n\n### Input:\n{inp}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instr}\n\n### Response:\n"
    return prompt, out

all_pairs = [format_pair(r) for r in ds["train"].select(range(3000))]
random.shuffle(all_pairs)
train_pairs, val_pairs = all_pairs[:2700], all_pairs[2700:]   # our own held-out split
print(f"train pairs: {len(train_pairs)} | val pairs: {len(val_pairs)}")

train pairs: 2700 | val pairs: 300


### **Hyperparameters:**

In [ ]:
EOS = tokenizer.eos_token_id
PAD = tokenizer.pad_token_id
if PAD is None:                              # SmolLM base tokenizer has no pad token
    tokenizer.pad_token = tokenizer.eos_token
    PAD = tokenizer.eos_token_id             # reuse EOS as the pad id
assert EOS is not None and PAD is not None, (EOS, PAD)
print("EOS:", EOS, "| PAD:", PAD)            # confirm both are real ints

MAX_LEN   = 512
batch_sz  = 4
lr, weight_decay, grad_clip = 2e-4, 0.01, 1.0
max_steps, eval_every = 500, 50

EOS: 0 | PAD: 0


### **Masked-example Builder:**

In [ ]:
def build_example(prompt_text, response_text):
    p = tokenizer(prompt_text,   add_special_tokens=False)["input_ids"]
    r = tokenizer(response_text, add_special_tokens=False)["input_ids"] + [EOS]
    input_ids = (p + r)[:MAX_LEN]
    labels    = ([-100]*len(p) + r)[:MAX_LEN]      # mask prompt → loss only on response
    return input_ids, labels

### **Collate SFT Batches (Packing) and Batch-Loader:**

In [ ]:
def collate(batch_pairs):
    ex = [build_example(p, r) for p, r in batch_pairs]
    maxlen = max(len(ids) for ids, _ in ex)
    input_ids, labels, attn = [], [], []
    for ids, lab in ex:
        pad = maxlen - len(ids)
        input_ids.append(ids + [PAD]  * pad)
        labels.append(   lab + [-100] * pad)       # padding never contributes
        attn.append(     [1]*len(ids) + [0]*pad)   # padding mask
    t = lambda z: torch.tensor(z).to(device)
    return t(input_ids), t(labels), t(attn)

def get_sft_batch(pool):
    return collate(random.sample(pool, batch_sz))

### **Optimizer + Fixed held-out Eval:**

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

@torch.no_grad()
def estimate_loss(batches=20):
    model.eval()
    total = 0.0
    for _ in range(batches):
        input_ids, labels, attn = get_sft_batch(val_pairs)
        total += model(input_ids=input_ids, attention_mask=attn, labels=labels).loss.item()
    model.train()
    return total / batches

### **Training Loop** (SFT: masked labels + attention_mask):

In [ ]:
model.train()
for step in range(max_steps):
    input_ids, labels, attn = get_sft_batch(train_pairs)
    loss = model(input_ids=input_ids, attention_mask=attn, labels=labels).loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if step % eval_every == 0 or step == max_steps - 1:
        print(f"step {step:4d} | train {loss.item():.4f} | val {estimate_loss():.4f}")

step    0 | train 2.1252 | val 2.0160
step   50 | train 1.6463 | val 1.6786
step  100 | train 2.1515 | val 1.7328
step  150 | train 1.9854 | val 1.7741
step  200 | train 2.2335 | val 1.7241
step  250 | train 1.2591 | val 1.6265
step  300 | train 1.3206 | val 1.8041
step  350 | train 1.4000 | val 1.6309
step  400 | train 2.3815 | val 1.7385
step  450 | train 1.5322 | val 1.6158
step  499 | train 1.9580 | val 1.6500


### **Generate:**

In [ ]:
def sft_generate(instruction, inp=""):
    prompt = (f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n"
              if inp.strip() else
              f"### Instruction:\n{instruction}\n\n### Response:\n")
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=150, do_sample=True,
                         temperature=0.7, top_p=0.9, repetition_penalty=1.3,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

print(sft_generate("Explain photosynthesis in simple terms."))

Photosynthesis is the process by which plants and other organisms convert light energy into chemical potential fuels called ATP, NADPH (nicotinamide adenine dinucleotide phosphate), or glucose to generate organic compounds such as oxygen gas for respiration


### **After SFT:**
Photosynthesis is the process by which plants and other photosynthetic organisms convert light energy into chemical potential from water to produce glucose (sugar) as a source of fuel for growth, development ,and maintenance .

### **Save The Adapters:**

In [ ]:
model.save_pretrained("smollm-lora-sft-alpaca")